In [ ]:
import collections
from collections import deque
import random

import pandas as pd
import numpy as np

# SWIM RIVER

![alt text](./static/river_swim.png "Title")

In [ ]:
class SwimRiver():
    def __init__(
        self, 
        steps: int,
        episode_length: int,
        total_states: int,
        dec: int = 3,
    ):
        self.steps = steps
        self.episode_length = episode_length
        self.total_states = total_states
        self.dec = dec
        self.states_count = None

    def get_action(self) -> int:
        """
        Return action from the given state. Actions are selected randomly
        :return: 0 means left, 1 is right
        """
        if self.state == 1:
            return 1
        if self.state == self.total_states:
            return 0

        return random.choice([0, 1])

    def do_action(self, action: int) -> int:
        """
        Simulate the action from the given state
        """
        if action == 0:
            return self.state - 1

        if self.state == 1:
            return random.choices([1, 2], weights=[0.4, 0.6])[0]

        delta = random.choices([-1, 0, 1], weights=[0.05, 0.6, 0.35])[0]
        return self.state + delta

    def run(self, show_results: bool = True):
        """
        Run the environment and count the frequencies
        """
        self.states_count = collections.Counter()
        self.state = 1
        episode_step = 0
        for _ in range(self.steps):
            action = self.get_action()
            self.state = self.do_action(action)
            self.states_count[self.state] += 1
            episode_step += 1
            if episode_step == self.episode_length:
                self.state = 1
                episode_step = 0

        if show_results:
            for state in range(1, self.total_states + 1):
                n_times = self.states_count[state]
                pct = round(100 * n_times/self.steps, self.dec)
                print(f"{state}:\t{n_times}\t->\t{pct}%")

In [ ]:
river = SwimRiver(
    steps=100000,
    episode_length=100000,
    total_states=6,
)
river.run()

In [ ]:
river = SwimRiver(
    steps=1000000,
    episode_length=1000000,
    total_states=20,
    dec=5,
)
river.run()

### Modified Get Action

In [ ]:
class ModSwimRiver(SwimRiver):
    def get_action(self) -> int:
        if self.state == 1:
            return 1
        if self.state == self.total_states:
            return 0

        n_times_prior = max(self.states_count[self.state - 1], 1)
        n_times_next = self.states_count[self.state + 1]
        total_n_times = n_times_prior + n_times_next
        w = [1 - n_times_prior/total_n_times, 1 - n_times_next/total_n_times]
        return random.choices([0, 1], weights=w)[0] 
        

In [ ]:
river = ModSwimRiver(
    steps=100000,
    episode_length=100000,
    total_states=6
)
river.run()

In [ ]:
river = ModSwimRiver(
    steps=1000000,
    episode_length=1000,
    total_states=20,
    dec=5,
)
river.run()

# GAMMA EFFECT

![alt text](./static/gamma.png "Title")

In [ ]:
def run_simulation(gamma, episodes):
    cum_reward = 0
    for i in range(int(episodes)):
        if i % 2:
            reward = 1
        else:
            reward = 2
        cum_reward += gamma ** i * reward
    print(f'cum_reward: {round(cum_reward, 4)}, final_discount: {gamma ** i}')
    return cum_reward

### ¿Quién gana?
#### 1) Gamma: 1 , Steps: 100
#### 2) Gamma: 0.99 , Steps: 1.000
#### 3) Gamma: 0.98 , Steps: 1.000.000

In [ ]:
run_simulation(1, 100)

In [ ]:
run_simulation(0.99, 1000)

In [ ]:
run_simulation(0.98, 10e6)

### ¿Es 0.95 alto?

In [ ]:
print(0.95 ** 50)
print(0.95 ** 100)
print(0.95 ** 200)

# DEQUE

In [ ]:
size = 1000000
arr_zeros = np.zeros(size)
list_ = list(arr_zeros)
deque_ = deque(list_, maxlen=size)

In [ ]:
%%timeit
if len(list_) == size:
    list_.pop(0)
    list_.append(0)

In [ ]:
%%timeit
list_.pop(0)
list_.append(0)

In [ ]:
%%timeit
deque_.append(0)

# .ILOC VS NUMPY

In [ ]:
size = int(10e5)
arr_ = np.random.uniform(0, 1, size=size)
nstep = 500
dict_ = {'col1': arr_, 'col2': arr_}
df = pd.DataFrame(dict_)
numpy_ = np.array([arr_, arr_])
col1 = dict_['col1']
double_dict = df.to_dict()

Comprobamos que accedemos al mismo valor

In [ ]:
print(df.iloc[nstep]['col1'])
print(df['col1'].iloc[nstep])
print(df['col1'].values[nstep])
print(numpy_[0][nstep])
print(dict_['col1'][nstep])
print(col1[nstep])
print(double_dict['col1'][nstep])

Medimos tiempos

In [ ]:
%%timeit
df.iloc[nstep]['col1']

In [ ]:
%%timeit
df['col1'].iloc[nstep]

In [ ]:
%%timeit
df['col1'].values[nstep]

In [ ]:
%%timeit
numpy_[0][nstep]

In [ ]:
%%timeit
dict_['col1'][nstep]

In [ ]:
%%timeit
col1[nstep]

In [ ]:
%%timeit
double_dict['col1'][nstep]

In [ ]:
numpy_[0]

In [ ]:
dict_['col1']

In [ ]:
double_dict['col1']